# CNN Model for Deep Learning - Face Mask Detection

In [1]:
#Importing the required libraries for modeling
import numpy as np
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
#importing DL libraries for CNN modeling
import tensorflow as tf
from tensorflow import keras
from keras_tqdm import TQDMNotebookCallback

Using TensorFlow backend.


Dataset containing 11800 pictures of Humans (With and Without Mask). We will use 10000 pictures for training, 800 for validation, and finally 1000 for testing

##### Assigning the Data Directories 

In [3]:
train_dir = r'C:\Users\Manish Gupta\Desktop\face_mask_detection\Train'
validation_dir = r'C:\Users\Manish Gupta\Desktop\face_mask_detection\Validation'
test_dir =r'C:\Users\Manish Gupta\Desktop\face_mask_detection\Test'

### Building the VGG19 Model first without Trainable Parameters for Faster Execution, TL Model

## Data Augmentation and Preprocessing

In [12]:
#Preprocessing as per given Data and Problem Specs
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
      rescale=1./255,
      rotation_range=40,
      width_shift_range=0.2,
      height_shift_range=0.2,
      shear_range=0.2,
      zoom_range=0.2,
      horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=(128, 128),
        batch_size=50,
        class_mode='binary')

validation_generator = test_datagen.flow_from_directory(
        validation_dir,
        target_size=(128, 128),
        batch_size=50,
        class_mode='binary')

Found 10000 images belonging to 2 classes.
Found 800 images belonging to 2 classes.


In [5]:
#Importing VGG19 with required Specs for the above problem
from tensorflow.keras.applications import VGG19

conv_base = VGG19(weights='imagenet',
                  include_top=False,
                  input_shape=(128, 128, 3))

80142336/80134624 [==============================] - 4s 0us/step

In [6]:
#Checking the conv-base model summary for VGG19
conv_base.summary()

Model: "vgg19"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_1 (InputLayer)         [(None, 128, 128, 3)]     0         
_________________________________________________________________
block1_conv1 (Conv2D)        (None, 128, 128, 64)      1792      
_________________________________________________________________
block1_conv2 (Conv2D)        (None, 128, 128, 64)      36928     
_________________________________________________________________
block1_pool (MaxPooling2D)   (None, 64, 64, 64)        0         
_________________________________________________________________
block2_conv1 (Conv2D)        (None, 64, 64, 128)       73856     
_________________________________________________________________
block2_conv2 (Conv2D)        (None, 64, 64, 128)       147584    
_________________________________________________________________
block2_pool (MaxPooling2D)   (None, 32, 32, 128)       0     

In [7]:
#Building the Reuired Model for the given Prolem using VGG19 Conv base, naming it as model_a for keras, using Sigmoid
#for Output Activation as Binary Classification is going on under the hood
from tensorflow.keras import models
from tensorflow.keras import layers

model_a = models.Sequential()
model_a.add(conv_base)
model_a.add(layers.Flatten())
model_a.add(layers.Dense(256, activation='relu'))
model_a.add(layers.Dense(1, activation='sigmoid'))

In [8]:
#Summarizing the above model
model_a.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
vgg19 (Model)                (None, 4, 4, 512)         20024384  
_________________________________________________________________
flatten (Flatten)            (None, 8192)              0         
_________________________________________________________________
dense (Dense)                (None, 256)               2097408   
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 257       
Total params: 22,122,049
Trainable params: 22,122,049
Non-trainable params: 0
_________________________________________________________________


#### Setting Trainable Params of COnvolution Base to False for Faster Training

In [9]:
# Now setting the Trainable Parameters of VGG Model to False thereby making the Training Execution faster
conv_base.trainable = False

In [10]:
# See the difference of 20M Trainable Params from Conv base are non-trainable now
model_a.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
vgg19 (Model)                (None, 4, 4, 512)         20024384  
_________________________________________________________________
flatten (Flatten)            (None, 8192)              0         
_________________________________________________________________
dense (Dense)                (None, 256)               2097408   
_________________________________________________________________
dense_1 (Dense)              (None, 1)                 257       
Total params: 22,122,049
Trainable params: 2,097,665
Non-trainable params: 20,024,384
_________________________________________________________________


### Compiling and Training the VGG19 Model

In [12]:
from tensorflow.keras import optimizers

model_a.compile(loss='binary_crossentropy',
              optimizer=optimizers.RMSprop(lr=2e-5),
              metrics=['acc'])

In [13]:
#Saving a h5 file for every epoch in training the VGG19 Model using callbacks just before training
checkpoint_cb = keras.callbacks.ModelCheckpoint("Face_Mask_Detection_Project_VGG19_Model-{epoch:02d}.h5")

In [ ]:
#Training the Model with given data and specs
history = model_a.fit_generator(
      train_generator,
      steps_per_epoch=200,  #for 10k training images with batch size of 50
      epochs=10,     #given 5-10 in the problem statement
      validation_data=validation_generator,
      validation_steps=16,    #for 800 Validation images
      callbacks=[checkpoint_cb])

Epoch 1/10
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
200/200 [==============================] - 6169s 31s/step - loss: 0.3127 - acc: 0.8908 - val_loss: 0.1082 - val_acc: 0.9650

### Reloading the Model from Saved Model File post Training

In [6]:
model_rl = keras.models.load_model("Face_Mask_Detection_Project_VGG19_Model-10.h5")

Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where


### Evaluating the Model

In [13]:
# Creating the Test Gen and Evaluating the Model
test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=(128, 128),
        batch_size=50,
        class_mode='binary')

Found 992 images belonging to 2 classes.


In [15]:
model_rl.evaluate(test_generator, steps=20) #20 Steps of 50 batches for 1k test images

20/20 [==============================] - 150s 8s/step - loss: 0.0322 - acc: 0.9889

[0.03219051583437249, 0.9889113]